## Authors
- David Robredo Manuel
- Duarte Novas Álvarez
- Rubén González Braña

## Data obtaintion

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd

2025-03-28 15:39:42.113991: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-28 15:39:42.131808: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743172782.148779  122243 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743172782.153955  122243 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-28 15:39:42.171718: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
#Returns a numpy array with size nrows x ncolumns-1. nrows and ncolums are the rows and columns of the dataset
#the Date column is skipped (ncolumns-1)
def readData(fname):
    with open(fname) as f:
        fileData = f.read()
  
    lines = fileData.split("\n")
    header = lines[0].split(",")
    lines = lines[1:] 
    #print(header) 
    #print("Data rows: ", len(lines))

    rawData = np.zeros((len(lines), len(header)-1)) #skip the Date column

    for i, aLine in enumerate(lines):       
        splittedLine = aLine.split(",")[:]
        rawData[i, 0] = splittedLine[0]
        rawData[i, 1:] = [float(x) for x in splittedLine[2:]] 

    return rawData

In [3]:
#Returns the train and test data, normalized. It also returns the standard deviation of Weekly_Sales
#Each list has a size equal to the number of stores
#For each store there is a list of size trainNSaples (testNSamples) x nColums-1 (the store id is skipped)
#Columns: Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
def splitTrainTest(rawData, testPercent):

    listStore = np.unique(rawData[:, 0])
    trainNSamples = np.zeros(len(listStore))
    
    for i, storeId in enumerate(listStore):
        trainNSamples[i] = np.count_nonzero(rawData[:, 0] == storeId)
    trainNSamples = np.floor((1-testPercent) *  trainNSamples)

    tmpTrain = np.zeros((int(np.sum(trainNSamples)), len(rawData[0])))

    store = -1
    counter = 0
    counterTrain = 0
    storeDict = dict(zip(listStore, trainNSamples))
    for i, aLine in enumerate(rawData):
        if store != aLine[0]:
            store = int(aLine[0])
            counter = 0
        if(counter < storeDict.get(store)):
            tmpTrain[counterTrain] = rawData[i][:]
            counterTrain += 1
            counter += 1

    meanData = tmpTrain.mean(axis=0)
    stdData = tmpTrain.std(axis=0)
    rawNormData = (rawData - meanData) / stdData

    allTrain = list()
    allTest = list()
    store = -1
    counter = 0
    for i, aLine in enumerate(rawNormData):
        splittedLine = [float(x) for x in aLine[1:]] #skip store id
        if store != rawData[i][0]:
            if i != 0:
                allTrain.append(storeDataTrain)
                allTest.append(storeDataTest)
            store = int(rawData[i][0])
            storeDataTrain = list()
            storeDataTest = list()
            counter = 0

        if(counter < storeDict.get(store)):
            storeDataTrain.append(splittedLine)
            counter += 1
        else:
            storeDataTest.append(splittedLine)

        if i == len(rawNormData)-1:
            allTrain.append(storeDataTrain)
            allTest.append(storeDataTest)

    return allTrain, allTest, stdData[1] #std of wSales

In [4]:
#generates a time series given the input and ouput data, the sequence length and the batch size
#seqLength is the number of weeks (observations) of data to be used as input
#the target will be the weekly sales in 2 weeks
def generateTimeSeries(data, wSales, seqLength, batchSize):   
    sampling_rate = 1 #keep all the data points 
    weeksInAdvance = 3
    delay = sampling_rate * (seqLength + weeksInAdvance - 1) #the target will be the weekly sales in 2 weeks
    
    dataset = keras.utils.timeseries_dataset_from_array(
        data[:-delay],
        targets=wSales[delay:],
        sampling_rate=sampling_rate,
        sequence_length=seqLength,
        shuffle=True,
        batch_size=batchSize,
        start_index=0)
    
    return dataset


In [5]:
def printTimeSeriesList(theList):
    print('list length', len(theList))
    print('First element')
    input, target = theList[0]
    print([float(x) for x in input.numpy().flatten()], [float(x) for x in target.numpy().flatten()])
    print('Last element')
    input, target = theList[-1]
    print([float(x) for x in input.numpy().flatten()], [float(x) for x in target.numpy().flatten()])

In [6]:
#returns the training and test time series
#it also returns the standard deviation of Weekly_Sales, and the number of input features
def generateTrainTestData(fileName, testPercent, seqLength, batchSize):
    rawData = readData(os.path.join(fileName))
    allTrain, allTest, stdSales = splitTrainTest(rawData, testPercent)
    
    for i in range(len(allTrain)):
        tmp_train = generateTimeSeries(np.array(allTrain[i]), np.array(allTrain[i])[:,0], seqLength, batchSize)
        tmp_test = generateTimeSeries(np.array(allTest[i]), np.array(allTest[i])[:,0], seqLength, batchSize)

        if i == 0:
            train_dataset = tmp_train
            test_dataset = tmp_test
        else:
            train_dataset = train_dataset.concatenate(tmp_train)
            test_dataset = test_dataset.concatenate(tmp_test)
            
    
    return train_dataset, test_dataset, stdSales, np.shape(allTrain)[2]

We established the seqLength to 12 to give models more information about past instance so they can learn more patterns within the data.

In [7]:
#generateTrainTestData(fileName, testPercent, seqLength, batchSize):
#trainData, testData: each element comes from keras.utils.timeseries_dataset_from_array, i.e., is a time series
#Columns: Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment

testPercent = 0.2
seqLength = 12
batchSize = 1
trainData, testData, stdSales, nFeatures = generateTrainTestData("datasets/walmart-sales-dataset-of-45stores.csv",testPercent, seqLength, batchSize) 

I0000 00:00:1743172790.827103  122243 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13760 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:17:00.0, compute capability: 7.5


### Implementing early stopping

In order to avoid overfitting, we implement early stopping to all models. Due to the characteristics of the problem, the fit function does not allow a validation dataset, therefore the early stopping will not perform as well as it is supposed to, but our intent is that it stops the training soon enough to reduce as much as possible the impact of overfitting.

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='mae', patience=2, restore_best_weights = True)

## Model creation 1

To start, we create a model architecture that mixes some of the most relevant RNN layers from keras. With this, we tried to build a somehow complex model so it can learn properly the dataset without overfitting (thanks to the early stopping)

In [ ]:
model1 = keras.Sequential([
    keras.layers.GRU(128, activation='tanh', return_sequences=True, input_shape=(seqLength, 6)),
    keras.layers.SimpleRNN(64, return_sequences=True),
    keras.layers.GRU(32, activation='tanh'),
    keras.layers.Dense(16),
    keras.layers.Dense(1)
])

model1.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [25]:
model1.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_6 (GRU)                     │ (None, 12, 128)        │        52,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 12, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,529 (291.13 KB)

 Trainable params: 74,529 (291.13 KB)

 Non-trainable params: 0 (0.00 B)

### Model training

In [26]:
model1.fit(trainData, epochs=20, callbacks=[early_stopping])

Epoch 1/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 65s 16ms/step - loss: 0.1916 - mae: 0.2736
Epoch 2/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.1019 - mae: 0.2073
Epoch 3/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0776 - mae: 0.1873
Epoch 4/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0620 - mae: 0.1701
Epoch 5/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0546 - mae: 0.1606
Epoch 6/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0491 - mae: 0.1543
Epoch 7/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0428 - mae: 0.1379
Epoch 8/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0432 - mae: 0.1442
Epoch 9/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0358 - mae: 0.1354
Epoch 10/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0326 - mae: 0.1299
Epoch 11/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss: 0.0297 - mae: 0.1238
Epoch 12/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - loss

### Model evaluation

After the training, we can see that the results are good but not enough since they are not below the objective (68000). Also, the test error is greater than the final training error.

In [12]:
testLoss1 = model1.evaluate(testData)

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0779 - mae: 0.2068


### Results

In [13]:
print("Standar deviation: ", stdSales, "Normalized test loss (MSE) and metrics (MAE): ", testLoss1)
print("Denormalized MAE: ", testLoss1[1] * stdSales)

Standar deviation:  571854.7800576452 Normalized test loss (MSE) and metrics (MAE):  [0.0490814633667469, 0.15906453132629395]
Denormalized MAE:  90961.81257657023


## Model creation 2

To try to improve past results, we are reducing the complexity of the model by eliminating the SimpleRNN and Dense layer, since the first architecture presented some overfitting.

In [ ]:
model2 = keras.Sequential([
    keras.layers.GRU(128, activation='tanh', return_sequences=True, input_shape=(seqLength, 6)),
    keras.layers.LSTM(16, return_sequences=True),
    keras.layers.GRU(16),
    keras.layers.Dense(1)
])

model2.compile(optimizer='adam', loss='mse', metrics=['mae'])

/home/ulc/cursos/curso281/.conda/envs/mia-dl2/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
model2.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 12, 128)        │        52,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 12, 16)         │         9,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 16)             │         1,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,153 (246.69 KB)

 Trainable params: 63,153 (246.69 KB)

 Non-trainable params: 0 (0.00 B)

### Model training

In [11]:
model2.fit(trainData, epochs=20, callbacks=[early_stopping])

Epoch 1/20


I0000 00:00:1743172824.528150  122278 cuda_dnn.cc:529] Loaded cuDNN version 90300


4005/4005 ━━━━━━━━━━━━━━━━━━━━ 24s 5ms/step - loss: 0.1837 - mae: 0.2548
Epoch 2/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 40s 5ms/step - loss: 0.0779 - mae: 0.1778    
Epoch 3/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 41s 5ms/step - loss: 0.0620 - mae: 0.1625
Epoch 4/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0503 - mae: 0.1506
Epoch 5/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0446 - mae: 0.1415
Epoch 6/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0355 - mae: 0.1311
Epoch 7/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0327 - mae: 0.1271
Epoch 8/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0351 - mae: 0.1282
Epoch 9/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0274 - mae: 0.1155
Epoch 10/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0222 - mae: 0.1063
Epoch 11/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0197 - mae: 0.1009
Epoch 12/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.0217 - mae: 0.1

### Model evaluation

After the training, it is clear that the final training error is much better than in previous attempt, and it is also reflected in the test error.

In [12]:
testLoss2 = model2.evaluate(testData)

180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0363 - mae: 0.1364    


### Results

In this attempt, we obtain an error below the objective (68000).

In [13]:
print("Standar deviation: ", stdSales, "Normalized test loss (MSE) and metrics (MAE): ", testLoss2)
print("Denormalized MAE: ", testLoss2[1] * stdSales)

Standar deviation:  571854.7800576452 Normalized test loss (MSE) and metrics (MAE):  [0.027539754286408424, 0.11431366950273514]
Denormalized MAE:  65370.81833106894


## Model creation 3

After the success in previous attempt, we try to change the architecture to see if it can be helpful. We remove the LSTM layer and add an extra Dense layer. 

In [ ]:
model3 = keras.Sequential([
    keras.layers.GRU(128, activation='tanh', return_sequences=True, input_shape=(seqLength, 6)),
    keras.layers.GRU(64, activation='tanh'),
    keras.layers.Dense(32),
    keras.layers.Dense(16),
    keras.layers.Dense(1)
])

model3.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [20]:
model3.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 12, 128)        │        52,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,097 (359.75 KB)

 Trainable params: 92,097 (359.75 KB)

 Non-trainable params: 0 (0.00 B)

### Model training

In [21]:
model3.fit(trainData, epochs=20, callbacks=[early_stopping])

Epoch 1/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 0.1518 - mae: 0.2529
Epoch 2/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0728 - mae: 0.1800
Epoch 3/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0596 - mae: 0.1632
Epoch 4/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0533 - mae: 0.1568
Epoch 5/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0454 - mae: 0.1428
Epoch 6/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0375 - mae: 0.1358
Epoch 7/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0306 - mae: 0.1222
Epoch 8/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0275 - mae: 0.1159
Epoch 9/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0286 - mae: 0.1174
Epoch 10/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0237 - mae: 0.1095
Epoch 11/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0214 - mae: 0.1048
Epoch 12/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0213 - m

### Model evaluation

The training error is more or less equal to the second attempt, but the test error shows a little improvement, obtaining a new record of the denormalized error of 60000.

In [22]:
testLoss3 = model3.evaluate(testData)

180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0265 - mae: 0.1262   


### Results

In [23]:
print("Standar deviation: ", stdSales, "Normalized test loss (MSE) and metrics (MAE): ", testLoss3)
print("Denormalized MAE: ", testLoss3[1] * stdSales)

Standar deviation:  571854.7800576452 Normalized test loss (MSE) and metrics (MAE):  [0.019704224541783333, 0.10545080900192261]
Denormalized MAE:  60302.5491886952


## Model creation 4

For this model, we wanted to return to the RNNs layers. Also, we change the optimizer in order to see if it affects positively to the results.

In [ ]:
model4 = keras.Sequential([
    keras.layers.GRU(128, activation='tanh', return_sequences=True, input_shape=(seqLength, 6)),
    keras.layers.LSTM(32, return_sequences=True),
    keras.layers.GRU(32),
    keras.layers.Dense(1)
])

model4.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

In [25]:
model4.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_6 (GRU)                     │ (None, 12, 128)        │        52,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 12, 32)         │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 32)             │         6,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,201 (309.38 KB)

 Trainable params: 79,201 (309.38 KB)

 Non-trainable params: 0 (0.00 B)

### Model training

In [26]:
model4.fit(trainData, epochs=20, callbacks=[early_stopping])

Epoch 1/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - loss: 0.1660 - mae: 0.2337
Epoch 2/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0939 - mae: 0.1853
Epoch 3/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0626 - mae: 0.1594
Epoch 4/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0516 - mae: 0.1454
Epoch 5/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0497 - mae: 0.1429
Epoch 6/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0397 - mae: 0.1293
Epoch 7/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0325 - mae: 0.1222
Epoch 8/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0279 - mae: 0.1143
Epoch 9/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0255 - mae: 0.1088
Epoch 10/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0228 - mae: 0.1030
Epoch 11/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0250 - mae: 0.1072
Epoch 12/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0206 - m

### Model evaluation

The training error seems similar to the second and third attempt, but the test error is much higher, probably due to overfitting.

In [27]:
testLoss4 = model4.evaluate(testData)

180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0513 - mae: 0.1699       


### Results

In [28]:
print("Standar deviation: ", stdSales, "Normalized test loss (MSE) and metrics (MAE): ", testLoss4)
print("Denormalized MAE: ", testLoss4[1] * stdSales)

Standar deviation:  571854.7800576452 Normalized test loss (MSE) and metrics (MAE):  [0.042096976190805435, 0.15700553357601166]
Denormalized MAE:  89784.36487094338


## Model creation 5

Finally, we wanted to try a combination of the main RNNs layers, returning also to the adam optimizer, to see if the results were good.

In [29]:
model5 = keras.Sequential([
    keras.layers.SimpleRNN(64, return_sequences=True, input_shape=(seqLength, 6)),
    keras.layers.LSTM(32, return_sequences=True),
    keras.layers.GRU(32),
    keras.layers.Dense(1)
])

model5.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [30]:
model5.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 12, 64)         │         4,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 12, 32)         │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_8 (GRU)                     │ (None, 32)             │         6,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,329 (91.13 KB)

 Trainable params: 23,329 (91.13 KB)

 Non-trainable params: 0 (0.00 B)

### Model training

In [31]:
model5.fit(trainData, epochs=20, callbacks=[early_stopping])

Epoch 1/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 55s 13ms/step - loss: 0.1848 - mae: 0.2592
Epoch 2/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0767 - mae: 0.1837
Epoch 3/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0542 - mae: 0.1566
Epoch 4/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0447 - mae: 0.1467
Epoch 5/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 54s 13ms/step - loss: 0.0315 - mae: 0.1269
Epoch 6/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0436 - mae: 0.1355
Epoch 7/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0292 - mae: 0.1248
Epoch 8/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0283 - mae: 0.1207
Epoch 9/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0257 - mae: 0.1172
Epoch 10/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 53s 13ms/step - loss: 0.0250 - mae: 0.1130
Epoch 11/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 55s 14ms/step - loss: 0.0249 - mae: 0.1119
Epoch 12/20
4005/4005 ━━━━━━━━━━━━━━━━━━━━ 55s 14ms/step - loss

### Model evaluation

As it can be seen, with this combination, the results are much worse than using only GRUs and LSTMs, not obtaining a test erro below 68000.

In [32]:
testLoss5 = model5.evaluate(testData)

180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0533 - mae: 0.1767


### Results

In [33]:
print("Standar deviation: ", stdSales, "Normalized test loss (MSE) and metrics (MAE): ", testLoss5)
print("Denormalized MAE: ", testLoss5[1] * stdSales)

Standar deviation:  571854.7800576452 Normalized test loss (MSE) and metrics (MAE):  [0.038585904985666275, 0.14752651751041412]
Denormalized MAE:  84363.7442235882


## Conclusions

From these experiments, we can determine that, for this data, a combination of GRUs and Dense layer is the best option to obtain the best possible model. Nevertheless, a combination of GRUs and LSTMs also obtain good results. Finally, we also deduce that by giving models more context information, by increasing the seqLength parameter, better results are obtained, since models have more information to work with.